
### 從影片變成img，每n張一數


In [13]:
import cv2
import os
import numpy as np

def extract_frames_every_n(input_video_path, output_folder, interval=5, frame_offset=0):
    """
    從影片中每隔指定間隔提取幀並儲存為圖片
    
    Args:
        input_video_path: 輸入影片路徑
        output_folder: 輸出資料夾路徑
        interval: 提取間隔（每幾幀提取一張）
        frame_offset: 檔名編號的偏移量（預設為0）
    
    Returns:
        成功儲存的圖片數量
    """
    # 確保輸出資料夾存在
    os.makedirs(output_folder, exist_ok=True)

    # 開啟影片
    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        raise ValueError(f"無法開啟影片：{input_video_path}")

    frame_count = 0
    saved_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % interval == 0:
            filename = f"VUE_{(frame_count + frame_offset):06d}.jpg"
            output_path = os.path.join(output_folder, filename)
            
            # 編碼並儲存圖片
            success, encoded_img = cv2.imencode('.jpg', frame)
            if success:
                try:
                    with open(output_path, 'wb') as f:
                        f.write(encoded_img.tobytes())
                    saved_count += 1
                except Exception as e:
                    raise IOError(f"寫入失敗 {filename}：{e}")
            else:
                raise ValueError(f"編碼失敗：{filename}")

        frame_count += 1

    cap.release()
    return saved_count

input_video = r"C:\Users\f1410\Desktop\vicon_chessbord_img\20260114收案\New Patient Classification\New Patient\New Session\chessboard_both_1.2129186.20260114161148.mp4"
output_base = r"C:\Users\f1410\Desktop\vicon_chessbord_img\20260114_chessboard_img\stereo\left\image"
extract_frames_every_n(input_video, output_base, interval=5)


2093

### 看角點圖片有沒有找好

In [ ]:
import numpy as np
import cv2
import glob
import os

def detect_chessboard_corners(image_folder, save_folder, chessboard_size=(9, 6), square_size=0.1):
    """
    檢測棋盤格角點並儲存結果圖片
    
    Args:
        image_folder: 輸入圖片資料夾路徑
        save_folder: 角點圖片儲存資料夾路徑
        chessboard_size: 棋盤格內角點數量 (寬, 高)
        square_size: 每個方格的實際大小（單位：根據實際測量）
    
    Returns:
        tuple: (obj_points, img_points) 用於相機標定的點集
    """
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
    
    # 準備棋盤格的3D點座標
    objp = np.zeros((chessboard_size[0] * chessboard_size[1], 3), np.float32)
    objp[:, :2] = np.mgrid[0:chessboard_size[0], 0:chessboard_size[1]].T.reshape(-1, 2) * square_size
    
    # 存儲所有圖像的3D點和2D點
    obj_points = []
    img_points = []
    
    # 讀取所有圖片
    images = glob.glob(os.path.join(image_folder, "*.jpg"))
    os.makedirs(save_folder, exist_ok=True)
    
    for fname in images:
        # 讀取圖片
        img = cv2.imread(fname)
        if img is None:
            # 嘗試用 numpy 讀取（避免中文路徑問題）
            try:
                with open(fname, 'rb') as f:
                    img_data = np.frombuffer(f.read(), np.uint8)
                    img = cv2.imdecode(img_data, cv2.IMREAD_COLOR)
            except Exception:
                continue
        
        if img is None:
            continue
        
        # 轉換為灰度圖
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # 找棋盤格角點
        ret, corners = cv2.findChessboardCorners(gray, chessboard_size, None)
        
        if ret:
            obj_points.append(objp)
            corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
            img_points.append(corners2)
            
            # 繪製角點
            cv2.drawChessboardCorners(img, chessboard_size, corners2, ret)
            
            # 儲存角點圖片
            base_name = os.path.basename(fname)
            save_path = os.path.join(save_folder, f"corners_{base_name}")
            success, encoded_img = cv2.imencode('.jpg', img)
            if success:
                with open(save_path, 'wb') as f:
                    f.write(encoded_img.tobytes())
    
    return obj_points, img_points

# 使用範例
image_folder = r"C:\Users\f1410\Desktop\vicon_chessbord_img\20260114_chessboard_img\stereo\left\image"
save_folder = r"C:\Users\f1410\Desktop\vicon_chessbord_img\20260114_chessboard_img\stereo\left\corner_image"
obj_points, img_points = detect_chessboard_corners(
    image_folder, 
    save_folder, 
    chessboard_size=(9, 6), 
    square_size=0.09
)

In [ ]:
import os
import shutil

# 定義資料夾路徑
chessboard_folder = r"C:\Users\f1410\Desktop\vicon_chessbord_img\20260114_chessboard_img\left\image"
corner_folder = r"C:\Users\f1410\Desktop\vicon_chessbord_img\20260114_chessboard_img\left\corner_image"
new_folder = r"C:\Users\f1410\Desktop\vicon_chessbord_img\20260114_chessboard_img\left\best_image"

# 確保新資料夾存在
os.makedirs(new_folder, exist_ok=True)

# 取得兩個資料夾中的檔案名稱
chessboard_files = set(os.listdir(chessboard_folder))
corner_files = set(os.listdir(corner_folder))

# 找出檔名後面編號相同的檔案
common_files = set()
for chessboard_file in chessboard_files:
    base_name, ext = os.path.splitext(chessboard_file)
    if ext == '.jpg':
        number = base_name.split('_')[-1]
        for corner_file in corner_files:
            corner_base_name, corner_ext = os.path.splitext(corner_file)
            if corner_ext == '.jpg' and corner_base_name.endswith(number):
                common_files.add(chessboard_file)
                break

# 列出相同編號的檔案並複製到新資料夾
print("相同編號的檔案有：")
for file in common_files:
    print(file)
    source_path = os.path.join(chessboard_folder, file)
    # 檢查目標檔案是否已存在，若存在則加上編號
    base_name, ext = os.path.splitext(file)
    destination_path = os.path.join(new_folder, file)
    counter = 1
    while os.path.exists(destination_path):
        destination_path = os.path.join(new_folder, f"{base_name}_{counter}{ext}")
        counter += 1
    shutil.copy2(source_path, destination_path)


### 產生參數，.npz檔

In [ ]:
import numpy as np
import cv2
import glob
import random
from tqdm import tqdm  # 引入進度條庫

criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
# 定義棋盤格的大小
chessboard_size = (9, 6)  # 這裡是 9x6 的內角點數量
square_size = 0.1  # 每個方格的實際大小（根據你的實際測量）

# 準備棋盤格的3D點座標
objp = np.zeros((chessboard_size[0] * chessboard_size[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:chessboard_size[0], 0:chessboard_size[1]].T.reshape(-1, 2) * square_size

# 存儲所有圖像的3D點和2D點
obj_points = []  # 3D點在世界座標系中的位置
img_points = []  # 2D點在圖像平面中的位置

# 讀取所有包含棋盤格的圖像
images = glob.glob(r"C:\Users\f1410\Desktop\vicon_chessbord_img\20250924_chessboard_img\left\selected_25\*.jpg*")
# selected_images = random.sample(images, min(1000, len(images)))

# 使用 tqdm 顯示圖像處理進度條
for fname in tqdm(images, desc="圖像處理進度", unit="張"):
    img = cv2.imread(fname)
    
    # 檢查圖片是否成功讀取
    if img is None:
        print(f"⚠️ 無法讀取圖片：{fname}")
        continue
    
    # 檢查圖片是否為空
    if img.size == 0:
        print(f"⚠️ 圖片為空：{fname}")
        continue
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 找到棋盤格的角點
    ret, corners = cv2.findChessboardCorners(gray, chessboard_size, None)     # 也可以用findChessboardCornersSB試試看，但不一定會更好
    # cv2.findChessboardCornersSB

    # 如果找到角點，則進行處理
    if ret:
        obj_points.append(objp)  # 添加3D點
        corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
        img_points.append(corners2)  # 添加2D點

        # 繪製並顯示角點
        cv2.drawChessboardCorners(img, chessboard_size, corners, ret)
        cv2.imshow('Corners', img)
        cv2.waitKey(50)

cv2.destroyAllWindows()

# 分批計算相機標定
batch_size = 50  # 每批計算的數量
total_batches = (len(obj_points) + batch_size - 1) // batch_size

print("正在進行相機標定計算...")

# 進度條初始化
pbar = tqdm(total=total_batches, desc="標定計算進度", unit="批")

# 分批標定計算
all_rvecs, all_tvecs = [], []
for i in range(0, len(obj_points), batch_size):
    batch_obj_points = obj_points[i:i + batch_size]
    batch_img_points = img_points[i:i + batch_size]

    try:
        ret, camera_matrix, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(
            batch_obj_points, batch_img_points, gray.shape[::-1], None, None
        )
        all_rvecs.extend(rvecs)
        all_tvecs.extend(tvecs)
        pbar.update(1)  # 更新進度條
    except Exception as e:
        print(f"標定計算錯誤：{e}")
        pbar.close()
        break

pbar.close()

# 打印結果
print("\n相機內部參數矩陣: \n", camera_matrix)
print("畸變係數: \n", dist_coeffs)
print("旋轉向量: \n", all_rvecs)
print("平移向量: \n", all_tvecs)

# 保存標定結果到檔案
np.savez('20250926_left_vue_25_chessboard.npz', camera_matrix=camera_matrix, dist_coeffs=dist_coeffs, rvecs=all_rvecs, tvecs=all_tvecs)
print("標定結果已保存")


查看參數

In [18]:
cam_param_L = np.load(r"C:\Users\f1410\Desktop\vicon_chessbord_img\20250924_chessboard_img\left\20250926_left_vue_25_chessboard.npz", allow_pickle=True)
camera_matrix_L = cam_param_L['camera_matrix']
dist_coeffs_L = cam_param_L['dist_coeffs']
print(f"camera_matrix: \n {camera_matrix_L}")
print(f"dist_coeffs: \n {dist_coeffs_L}")

camera_matrix: 
 [[1.33083633e+03 0.00000000e+00 1.00501442e+03]
 [0.00000000e+00 1.32573638e+03 6.40334493e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]
dist_coeffs: 
 [[-0.42404383  0.22987828 -0.00526783 -0.0020229  -0.06730389]]


In [20]:
cam_param_R = np.load(r"C:\Users\f1410\Desktop\vicon_chessbord_img\20250924_chessboard_img\right\20250926_right_vue_25_chessboard.npz", allow_pickle=True)
camera_matrix_R = cam_param_R['camera_matrix']
dist_coeffs_R = cam_param_R['dist_coeffs']
print(f"camera_matrix: \n {camera_matrix_R}")
print(f"dist_coeffs: \n {dist_coeffs_R}")

camera_matrix: 
 [[1.39108177e+03 0.00000000e+00 8.98150247e+02]
 [0.00000000e+00 1.37886200e+03 6.80967547e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]
dist_coeffs: 
 [[-0.40907323  0.20942974 -0.00905899  0.00173602 -0.06212284]]


### 用上面產生的parameter，校正看看 </br>

In [ ]:
import numpy as np
import cv2
# -------------------------- 畸變校正 ---------------------------------
'''
姵甄有個沒有切的版本，之後問她
input : (1)畸變前影片、(2).npz檔案
output : 校正後(直線)的圖片
'''
cam_param = np.load(r"C:\Users\f1410\Desktop\vicon_chessbord_img\20250924_chessboard_img\left\20250926_left_vue_25_chessboard.npz", allow_pickle=True)
camera_matrix = cam_param['camera_matrix']
dist_coeffs = cam_param['dist_coeffs']

def calibration(un_cal_frame,w,h):
    # 計算新的相機矩陣
    new_camera_matrix, roi = cv2.getOptimalNewCameraMatrix(camera_matrix, dist_coeffs, (w, h), 0, (w, h))

    # 矯正畸變
    dst = cv2.undistort(un_cal_frame, camera_matrix, dist_coeffs, None, new_camera_matrix)
    
    # 檢查 ROI 並裁剪圖像
    if roi is not None and all(roi):
        x, y, w, h = roi
        frame = dst[y:y+h, x:x+w]
    else:
        frame = dst  # 如果沒有有效 ROI，就用整張圖
    return frame

cap = cv2.VideoCapture(r"C:\Users\f1410\Desktop\vicon_chessbord_img\20250819_棋盤格\左VUE\mult_vue_chessboard_L_1.2129186.20250819164536.mp4")

# 輸出影像設定
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
ret, img = cap.read()
img = calibration(img,width,height)
cv2.imwrite('40chessboard_calibration_0819_left_vue_1.jpg', img)

input : 未校正圖片、npz檔 </br>
iutput : 校正後圖片

In [21]:
def calibration(un_cal_frame, w, h):
    new_camera_matrix, roi = cv2.getOptimalNewCameraMatrix(camera_matrix_R, dist_coeffs_R, (w, h), 0, (w, h))
    dst = cv2.undistort(un_cal_frame, camera_matrix_R, dist_coeffs_R, None, new_camera_matrix)

    if roi is not None and all(roi):
        x, y, w, h = roi
        frame = dst[y:y+h, x:x+w]
    else:
        frame = dst  # 如果沒有有效 ROI，就用整張圖

    return frame


cap = cv2.VideoCapture(r"C:\Users\f1410\Desktop\vicon_chessbord_img\20250924_chessboard_img\court\vue_chessboard_court_04.2129991.20250924133935_frame_0010_t0.2s.jpg")  

# 輸出影像設定
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
ret, img = cap.read()
img = calibration(img,width,height)
cv2.imwrite('chessboard_calibration_0926_right_vue.jpg', img)

True